Import thư viện:

In [3]:
from pathlib import Path
import pandas as pd
from plotnine import (
ggplot,
aes,
geom_histogram,
geom_bar,
geom_col,
facet_wrap,
labs,
)

Kiểm tra nhanh:

In [4]:
data_dir = Path("data")
county = pd.read_csv(data_dir / "county.csv")
run10 = pd.read_csv(data_dir / "run10.csv")
exam_grades = pd.read_csv(data_dir / "exam_grades.csv")
loans = pd.read_csv(data_dir / "loans_full_schema.csv")
ncbirths = pd.read_csv(data_dir / "ncbirths.csv")
print("county:", county.shape)
print("run10:", run10.shape)
print("exam_grades:", exam_grades.shape)
print("loans_full_schema:", loans.shape)
print("ncbirths:", ncbirths.shape)

county: (3142, 15)
run10: (16924, 9)
exam_grades: (233, 6)
loans_full_schema: (10000, 55)
ncbirths: (1000, 13)


Phần A — Tính tay

Bài 1:

1. Các hàng còn lại sau .query()
Điều kiện lọc là bill_length_mm >= 45 VÀ body_mass_g < 5000.
Các ID thỏa mãn và được giữ lại là: 5, 7, 9, 10, 11, 12.
(Các ID 1, 2, 3, 4 bị loại vì bill_length < 45; các ID 6, 8 bị loại vì body_mass >= 5000).

2. Tạo cột heavy = True/False cho các hàng còn lại
Điều kiện gán là body_mass_g > 4000:

ID 5 (4500g): True
ID 7 (4450g): True
ID 9 (3500g): False
ID 10 (3900g): False
ID 11 (3650g): False
ID 12 (3525g): False

3. Chia nhóm theo species và tính toán (groupby & agg)
Bộ dữ liệu còn lại gồm 2 loài. Bỏ qua loài Adelie vì không còn quan sát nào (observed=True).

*Nhóm Gentoo (ID 5, 7):
-n = 2 (có 2 quan sát)
-p_heavy = Trung bình của (True, True) = (1 + 1) / 2 = 1.0 (hoặc 100%)
-mean_depth = (13.2 + 14.1) / 2 = 13.65

*Nhóm Chinstrap (ID 9, 10, 11, 12):
-n = 4 (có 4 quan sát)
-p_heavy = Trung bình của (False, False, False, False) = (0 + 0 + 0 + 0) / 4 = 0.0 (hoặc 0%)
-mean_depth = (17.9 + 19.5 + 19.2 + 18.7) / 4 = 75.3 / 4 = 18.825

4. Giải thích trung bình của True/False
Trong máy tính, True được hiểu là 1 và False là 0, do đó khi tính trung bình (tổng các giá trị chia cho tổng số lượng), ta đang lấy "số lượng các trường hợp True" chia cho "tổng số trường hợp", và đây chính xác là công thức tính tỉ lệ.

5. Nếu đổi thứ tự groupby() và .query()
Câu hỏi thống kê không còn giống hệt.
Khi chạy .query() trước, ta đang lọc dữ liệu ở cấp độ cá thể (chỉ giữ lại những con chim cánh cụt thỏa mãn điều kiện để đem đi phân tích). Nếu ta gom nhóm (groupby) và tổng hợp (agg) trước rồi mới lọc, điều kiện lọc sẽ áp dụng lên cấp độ tổng hợp của loài (ví dụ: tìm những loài có cân nặng trung bình < 5000), làm thay đổi hoàn toàn bản chất câu hỏi từ đặc điểm cá thể sang đặc điểm nhóm.

Bài 2

1. Số phần trăm cá thể trong 1 khoảng
[2.7, 3.5): 71/342 * 100% ~ 20.76%
[3.5, 4.0): 94/342 * 100% ~ 27.49%
[4.0, 5.0): 110/342 * 100% ~ 32.16%
[5.0, 6.4]: 67/342 * 100% ~ 19.59%

2. Chiều cao histogram theo thang
[2.7, 3.5):
-chiều rộng lớp: 3.5 - 2.7 = 0.8 (kg)
-mật độ: 20.76 / 0.8 ~ 25.95 (%/kg) 
[3.5, 4.0):
-chiều rộng lớp: 4.0 - 3.5 = 0.5 (kg)
-mật độ: 27.49 / 0.5 ~ 54.98 (%/kg) 
[4.0, 5.0):
-chiều rộng lớp: 5.0 - 4.0 = 1.0 (kg)
-mật độ: 32.16 / 1.0 = 32.16 (%/kg) 
[5.0, 6.4]:
-chiều rộng lớp: 6.4 - 5.0 = 1.4 (kg)
-mật độ: 19.59 / 1.4 ~ 13.99 (%/kg) 

4. 
* Khoảng có nhiều cá thể nhất: Khoảng [4.0, 5.0) với 110 cá thể (chiếm 32.16%).
* Khoảng dày đặc nhất trên mỗi kg: Khoảng [3.5, 4.0) với mật độ cao nhất là 54.98 %/kg.
* Hai câu trả lời KHÔNG giống nhau. Một khoảng có thể chứa ít cá thể hơn tổng thể, nhưng nếu khoảng đó rất hẹp thì mật độ của nó vẫn sẽ rất cao

5. Nhận xét lỗi sai của bạn học sinh
Bạn học sinh đã nhầm lẫn giữa Chiều cao của cột (Mật độ) và Diện tích của cột (Số lượng/Tỉ lệ phần trăm).Cột [3.5, 4.0) cao hơn cột [4.0, 5.0) chỉ vì khoảng giá trị của nó ngắn hơn hẳn (0.5 kg so với 1.0 kg), khiến các cá thể bị "dồn nén" lại trong một khoảng hẹp. Trên thực tế, cột [4.0, 5.0) rộng gấp đôi nên dù thấp hơn, tổng diện tích của nó (tương ứng số lượng chim) vẫn lớn hơn (110 cá thể so với 94 cá thể).

Bài 3

1 & 2. Dựng hai danh sách A và B
Dữ liệu yêu cầu là các số nguyên. Chú ý ký hiệu khoảng [a, b) nghĩa là lấy từ a đến sát b nhưng không bao gồm b. Do đó, số nguyên lớn nhất trong khoảng [a, b) là b - 1.

Dựa vào tần suất 5, 5, 1, 1 cho các lớp:

Lớp [3000, 3500) có 5 quan sát:
Gần mép trái nhất (Danh sách A): 3000, 3000, 3000, 3000, 3000
Gần mép phải nhất (Danh sách B): 3499, 3499, 3499, 3499, 3499

Lớp [3500, 4000) có 5 quan sát:
Danh sách A: 3500, 3500, 3500, 3500, 3500
Danh sách B: 3999, 3999, 3999, 3999, 3999

Lớp [4000, 4500) có 1 quan sát:
Danh sách A: 4000
Danh sách B: 4499

Lớp [4500, 5000) có 1 quan sát:
Danh sách A: 4500
Danh sách B: 4999

Danh sách A: [3000, 3000, 3000, 3000, 3000, 3500, 3500, 3500, 3500, 3500, 4000, 4500]

Danh sách B: [3499, 3499, 3499, 3499, 3499, 3999, 3999, 3999, 3999, 3999, 4499, 4999]
(Cả hai danh sách này khi vẽ histogram với các lớp trên đều cho ra đúng tần suất 5, 5, 1, 1).

3. Ba đặc điểm của dữ liệu gốc bị mất đi (không dùng trung bình hay độ lệch chuẩn)
Histogram là phép nén dữ liệu (gom nhóm thành các thùng). Khi nhìn vào biểu đồ, ta không thể khôi phục các đặc điểm sau:

Giá trị lớn nhất (Max) và nhỏ nhất (Min) chính xác: Ta chỉ biết Min nằm đâu đó trong đoạn [3000, 3500) và Max nằm trong [4500, 5000). Ở Danh sách A, Max là 4500; ở Danh sách B, Max là 4999.

Trung vị (Median) chính xác: Dù biết trung vị rơi vào lớp [3500, 4000) (vì tổng là 12, trung vị nằm giữa giá trị thứ 6 và 7), ta không thể biết con số chính xác của trung vị là bao nhiêu.

Khoảng biến thiên thực sự (Range): Vì không biết Min và Max chính xác, ta không thể tính được chênh lệch Max - Min.

Sự phân bố cụ thể bên trong một lớp: Các giá trị có thể tụ tập hết ở mép trái, mép phải, hoặc rải đều trong lớp. Histogram không thể hiện điều này.

4. Phản biện phát biểu của sinh viênSinh viên nói: "Nếu hai bộ dữ liệu có cùng histogram thì về mặt thống kê chúng là một."
Phản biện: Phát biểu này là sai. 
Hãy nhìn trực tiếp vào Danh sách A và Danh sách B vừa lập ở trên: Cả hai đều cho ra cùng một biểu đồ histogram (5, 5, 1, 1). Tuy nhiên, mọi giá trị trong Danh sách B đều nặng hơn Danh sách A đúng 499 gram. Đối với loài chim cánh cụt (nặng cỡ 3-5kg), chênh lệch 500g (khoảng 10-15% trọng lượng cơ thể) là một sự khác biệt sinh học khổng lồ (ví dụ: chim suy dinh dưỡng so với chim trưởng thành khỏe mạnh). Trung vị của A là 3500, trong khi trung vị của B lên tới 3999.Như vậy, hai bộ dữ liệu có đặc tính thực tế và thống kê hoàn toàn khác nhau, nhưng lại bị "cào bằng" thành một hình ảnh giống hệt nhau khi vẽ histogram.

Bài 4

1. Phần trăm loài trong mỗi khoảng
[1.9, 5): 11/83 * 100% ~ 13.25%
[5, 8): 10/83 * 100% ~ 12.05%
[8, 12): 31/83 * 100% ~ 37.35%
[12, 20]: 31/83 * 100% ~ 37.35%

2. Mật độ “% trên mỗi giờ” cho mỗi khoảng
[1.9, 5): 
-chiều rộng lớp: 5 - 1.9 = 3.1 (giờ)
-mật độ: 13.25 / 3.1 ~ 4.28 (%/giờ)
[5, 8):
-chiều rộng lớp: 8 - 5 = 3.0 (giờ)
-mật độ: 12.05 / 3.0 ~ 4.02 (%/giờ)
[8, 12):
-chiều rộng lớp: 12 - 8 = 4.0 (giờ)
-mật độ: 37.35 / 4.0 ~ 9.34 (%/giờ)
[12, 20]:
-chiều rộng lớp: 20 - 12 = 8.0 (giờ)
-mật độ: 37.35 / 8.0 ~ 4.67 (%/giờ)

3. So sánh chiều cao của hai khoảng cuối
Cùng có 31 loài, nhưng trong Histogram mật độ, hai khoảng này KHÔNG cao bằng nhau.
Chiều cao của cột biểu thị mật độ (tỉ lệ % trên mỗi đơn vị độ rộng), không biểu thị tổng số lượng loài.
Khoảng [8, 12) có độ rộng 4 giờ => Chiều cao mật độ = 9.34 %/giờ.
Khoảng [12, 20] có độ rộng tới 8 giờ (rộng gấp đôi) => Chiều cao mật độ = 4.67 %/giờ (chỉ cao bằng một nửa khoảng [8, 12) ).

4. Nhận xét về số loài ngủ đúng 9 giờ vs 15 giờ
KHÔNG thể kết luận “có nhiều loài ngủ đúng 9 giờ hơn đúng 15 giờ”.
Giải thích:
Dữ liệu đã bị nén/gom nhóm vào các khoảng. Ta chỉ biết có 31 loài ngủ trong khoảng [8, 12) và 31 loài trong khoảng [12, 20]. Dữ liệu không cho biết phân bố chi tiết bên trong mỗi khoảng:
Tất cả 31 loài trong khoảng [8, 12) hoàn toàn có thể rơi vào 8.5 giờ hoặc 11 giờ (không loài nào ngủ đúng 9 giờ).
Tương tự, nhiều loài trong khoảng [12, 20] có thể tập trung vào đúng 15 giờ.
Thời gian ngủ là biến liên tục, nên từ bảng tần số nhóm không thể suy ra số lượng cá thể chính xác tại một điểm cụ thể.

5. Phân biệt Bar Chart (biểu đồ cột) và Histogram liên tục
Đồ họa vẽ 4 cột bằng chiều rộng nhau chỉ phù hợp làm Bar Chart đại diện cho 4 "nhãn nhóm" phân loại, nhưng KHÔNG ĐƯỢC đọc như một Histogram trên trục số liên tục vì:
-Bóp méo khoảng cách số: Độ rộng thực tế của các khoảng trên trục số thực là rất khác nhau (3.1; 3.0; 4.0; 8.0). Nếu vẽ các cột rộng bằng nhau, khoảng [12, 20] (rộng 8 đơn vị) sẽ bị ép nhỏ bằng khoảng [5, 8) (chỉ rộng 3 đơn vị).
-Vi phạm nguyên tắc diện tích (Area Principle): Trên trục số liên tục, diện tích hình chữ nhật biểu thị tỉ lệ/số lượng dữ liệu (Diện tích = Độ rộng * Chiều cao). Khi vẽ các cột rộng bằng nhau, diện tích của cột [12, 20] và [8, 12) sẽ bằng nhau (vì cùng có số lượng 31), làm người xem hiểu sai rằng mật độ phân bố dữ liệu trên trục số của hai khoảng này là như nhau.

Bài 5

1. Tính lại các tỉ lệ từ tử số và mẫu số
*Adelie: p1 = 35 / 151 ~ 23.18%
*Chinstrap: p2 = 15 / 68 ~ 22.06%
*Gentoo: p3 = 122 / 123 ~ 99.19%
*Tỉ lệ gộp (Biên): p = (35 + 15 + 122) / (151 + 68 + 123) = 172 / 342 ~ 50.29%

2. Chứng minh $50.29\%$ là trung bình có trọng số của ba tỉ lệ theo loài  
Gọi $n_1 = 151, n_2 = 68, n_3 = 123$ là số quan sát của từng loài, và $N = n_1 + n_2 + n_3 = 342$ là tổng số quan sát. Trọng số $w_i$ của mỗi loài là tỉ lệ đại diện của loài đó trong mẫu: $w_i = \frac{n_i}{N}$.   
Công thức trung bình có trọng số của ba tỉ lệ $p_1, p_2, p_3$:$$\bar{p} = w_1 p_1 + w_2 p_2 + w_3 p_3$$
Thay công thức $p_i = \frac{k_i}{n_i}$ vào:$$\bar{p} = \left(\frac{n_1}{N}\right) \cdot \left(\frac{k_1}{n_1}\right) + \left(\frac{n_2}{N}\right) \cdot \left(\frac{k_2}{n_2}\right) + \left(\frac{n_3}{N}\right) \cdot \left(\frac{k_3}{n_3}\right)$$
$$\bar{p} = \frac{k_1 + k_2 + k_3}{N} = \frac{35 + 15 + 122}{342} = \frac{172}{342} \approx \mathbf{50.29\%}$$
(Điều phải chứng minh)

3. Đánh giá kết luận của báo cáo  
Kết luận của báo cáo là SAI HOÀN TOÀN. Tỉ lệ 50.3% không hỗ trợ cho phát biểu đó.  
Giải thích: Biến species có khả năng dự đoán cực kỳ mạnh về việc chim cánh cụt có nặng trên 4000 g hay không:Nếu là loài Gentoo, xác suất nặng > 4000 g gần như tuyệt đối (~ 99.2%).Nếu là Adelie hoặc Chinstrap, xác suất nặng > 4000 g lại rất thấp (chỉ khoảng 22% - 23%).Tỉ lệ 50.3% chung thực chất chỉ là "ngẫu nhiên" do số lượng chim Gentoo (123 con) xấp xỉ tổng số chim Adelie + Chinstrap (151 + 68 = 219 con) gộp lại.

4. Ước tính tỉ lệ biên $H=1$ cho Mẫu 1 ($10$ Adelie, $10$ Chinstrap, $80$ Gentoo)  
Số lượng cá thể dự kiến có $H=1$ trong $100$ con:  
-Adelie: $10 \times \frac{35}{151} \approx 2.32$  
-Chinstrap: $10 \times \frac{15}{68} \approx 2.21$  
-Gentoo: $80 \times \frac{122}{123} \approx 79.35$  
Tổng số cá thể $H=1$:$$2.32 + 2.21 + 79.35 = 83.88$$  
Tỉ lệ biên mới: $\frac{83.88}{100} = \mathbf{83.88\%}$

5. Ước tính tỉ lệ biên $H=1$ cho Mẫu 2 ($45$ Adelie, $45$ Chinstrap, $10$ Gentoo)  
Số lượng cá thể dự kiến có $H=1$ trong $100$ con:  
-Adelie: $45 \times \frac{35}{151} \approx 10.43$  
-Chinstrap: $45 \times \frac{15}{68} \approx 9.93$  
-Gentoo: $10 \times \frac{122}{123} \approx 9.92$  
Tổng số cá thể $H=1$:$$10.43 + 9.93 + 9.92 = 30.28$$  
Tỉ lệ biên mới: $\frac{30.28}{100} = \mathbf{30.28\%}$  
Ý nghĩa của hai kết quả:  
Dù tỉ lệ theo từng loài (Xác suất có điều kiện $P(H=1 \mid \text{species})$) không hề thay đổi, tỉ lệ biên chung $P(H=1)$ lại biến động mạnh từ $30.28\%$ đến $83.88\%$ tùy thuộc vào cơ cấu nhóm. Tỉ lệ biên chịu ảnh hưởng lớn bởi trọng số (quy mô) của các phân nhóm.

Phần B -- Freedman

Khởi tạo python:

In [5]:
from pathlib import Path
import pandas as pd
from plotnine import (
ggplot,
aes,
geom_histogram,
geom_bar,
geom_col,
facet_wrap,
labs,
)
data_dir = Path("data")

Bài 6

In [6]:
county = pd.read_csv("data/county.csv")
print(county)

                   name    state   pop2000  pop2010   pop2017  pop_change  \
0        Autauga County  Alabama   43671.0    54571   55504.0        1.48   
1        Baldwin County  Alabama  140415.0   182265  212628.0        9.19   
2        Barbour County  Alabama   29038.0    27457   25270.0       -6.22   
3           Bibb County  Alabama   20826.0    22915   22668.0        0.73   
4         Blount County  Alabama   51024.0    57322   58013.0        0.68   
...                 ...      ...       ...      ...       ...         ...   
3137  Sweetwater County  Wyoming   37613.0    43806   43534.0       -3.57   
3138       Teton County  Wyoming   18251.0    21294   23265.0        4.21   
3139       Uinta County  Wyoming   19742.0    21118   20495.0       -2.22   
3140    Washakie County  Wyoming    8289.0     8533    8064.0       -4.31   
3141      Weston County  Wyoming    6644.0     7208    6927.0       -2.93   

      poverty  homeownership  multi_unit  unemployment_rate metro  \
0     